# AI_PARSE_DOCUMENT — Invoice Processing

`AI_PARSE_DOCUMENT` extracts structured data from documents (PDF, images) on a Snowflake stage.

| Item | Detail |
|---|---|
| **Function** | `SNOWFLAKE.CORTEX.AI_PARSE_DOCUMENT` |
| **Data** | PDF invoice files on `@invoice_pdfs` stage |
| **Use-case** | Automated invoice data extraction and processing |

> **Note:** This lab requires PDF files to be uploaded to the `@invoice_pdfs` stage.
> You can upload sample invoices through the Snowsight UI or use PUT commands.


## Step 1 — Environment & Stage Setup

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

In [ ]:
CREATE STAGE IF NOT EXISTS invoice_pdfs
    COMMENT = 'Invoice PDF files for AI_PARSE_DOCUMENT demo';

CREATE OR REPLACE TABLE invoice_results (
    file_name    VARCHAR(200),
    parsed_data  VARIANT,
    processed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

## Step 2 — Upload Sample Invoices

Upload PDF invoices to the stage. You can:
1. Use the Snowsight **Add Data** wizard to upload files to `@invoice_pdfs`
2. Use a PUT command from SnowSQL: `PUT file:///path/to/invoice.pdf @invoice_pdfs`

Verify uploaded files:

In [ ]:
LIST @invoice_pdfs;

## Step 3 — Parse Invoice Documents

Use `AI_PARSE_DOCUMENT` with `TO_FILE()` to reference staged files.
The function extracts text and structure from the document.

In [ ]:
-- Parse using OCR mode (extracts raw text)
SELECT
    RELATIVE_PATH AS file_name,
    SNOWFLAKE.CORTEX.AI_PARSE_DOCUMENT(
        TO_FILE(@invoice_pdfs, RELATIVE_PATH),
        {'mode': 'OCR'}
    ) AS parsed_content
FROM DIRECTORY(@invoice_pdfs)
LIMIT 5;

In [ ]:
-- Parse using LAYOUT mode (preserves document structure)
SELECT
    RELATIVE_PATH AS file_name,
    SNOWFLAKE.CORTEX.AI_PARSE_DOCUMENT(
        TO_FILE(@invoice_pdfs, RELATIVE_PATH),
        {'mode': 'LAYOUT'}
    ) AS parsed_content
FROM DIRECTORY(@invoice_pdfs)
LIMIT 5;

## Step 4 — Extract Structured Fields with COMPLETE

Combine `AI_PARSE_DOCUMENT` with `COMPLETE` to extract specific invoice fields.

In [ ]:
WITH parsed AS (
    SELECT
        RELATIVE_PATH AS file_name,
        SNOWFLAKE.CORTEX.AI_PARSE_DOCUMENT(
            TO_FILE(@invoice_pdfs, RELATIVE_PATH),
            {'mode': 'LAYOUT'}
        ):content::STRING AS doc_text
    FROM DIRECTORY(@invoice_pdfs)
)
SELECT
    file_name,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-3-5-haiku',
        'Extract from this invoice and return JSON with keys: invoice_number, date, vendor_name, total_amount, line_items (array). Document: ' || doc_text
    ) AS extracted_fields
FROM parsed;

## Key Takeaways

- `AI_PARSE_DOCUMENT` supports PDF and image files via `TO_FILE()` references.
- Two modes: `OCR` (raw text extraction) and `LAYOUT` (preserves structure).
- Chain with `COMPLETE` to extract specific fields from parsed content.
- Ideal for invoices, contracts, forms, and any document processing pipeline.
- Files must be on a Snowflake stage — use PUT or Snowsight upload.
